# DS605 Lab 5 — Part A: Scikit-learn

### Imports

In [1405]:
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report)

np.random.seed(42)

### Load

In [1406]:
df = pd.read_csv("garments_worker_productivity.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (1197, 15)
Columns: ['date', 'quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'actual_productivity']


### Data

In [1407]:
df.info()
df.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 1197 entries, 0 to 1196
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   1197 non-null   str    
 1   quarter                1197 non-null   str    
 2   department             1197 non-null   str    
 3   day                    1197 non-null   str    
 4   team                   1197 non-null   int64  
 5   targeted_productivity  1197 non-null   float64
 6   smv                    1197 non-null   float64
 7   wip                    691 non-null    float64
 8   over_time              1197 non-null   int64  
 9   incentive              1197 non-null   int64  
 10  idle_time              1197 non-null   float64
 11  idle_men               1197 non-null   int64  
 12  no_of_style_change     1197 non-null   int64  
 13  no_of_workers          1197 non-null   float64
 14  actual_productivity    1197 non-null   float64
dtypes: float64(6), 

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382
5,1/1/2015,Quarter1,sweing,Thursday,7,0.80,25.90,984.0,6720,38,0.0,0,0,56.0,0.800125
6,1/1/2015,Quarter1,finishing,Thursday,2,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.755167
7,1/1/2015,Quarter1,sweing,Thursday,3,0.75,28.08,795.0,6900,45,0.0,0,0,57.5,0.753683
8,1/1/2015,Quarter1,sweing,Thursday,2,0.75,19.87,733.0,6000,34,0.0,0,0,55.0,0.753098
9,1/1/2015,Quarter1,sweing,Thursday,1,0.75,28.08,681.0,6900,45,0.0,0,0,57.5,0.750428


### Missing values

In [1408]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({"Missing": missing, "Percent": missing_pct.round(2)})
print(missing_df[missing_df["Missing"] > 0])

     Missing  Percent
wip      506    42.27


**Only `wip` has missing values** — 506 out of 1197 rows (~42%).

That's too much to drop (we'd lose nearly half the data). We'll impute instead. The 42% will also justify adding a `wip_missing` flag later.

### Categorical inspection

In [1409]:
cat_cols = ["date", "quarter", "department", "day", "team"]
for col in cat_cols:
    print(f"\n--- {col} ---")
    print("Unique:", df[col].nunique())
    print(df[col].unique())


--- date ---
Unique: 59
<ArrowStringArray>
[ '1/1/2015',  '1/3/2015',  '1/4/2015',  '1/5/2015',  '1/6/2015',  '1/7/2015',
  '1/8/2015', '1/10/2015', '1/11/2015', '1/12/2015', '1/13/2015', '1/14/2015',
 '1/15/2015', '1/17/2015', '1/18/2015', '1/19/2015', '1/20/2015', '1/21/2015',
 '1/22/2015', '1/24/2015', '1/25/2015', '1/26/2015', '1/27/2015', '1/28/2015',
 '1/29/2015', '1/31/2015',  '2/1/2015',  '2/2/2015',  '2/3/2015',  '2/4/2015',
  '2/5/2015',  '2/7/2015',  '2/8/2015',  '2/9/2015', '2/10/2015', '2/11/2015',
 '2/12/2015', '2/14/2015', '2/15/2015', '2/16/2015', '2/17/2015', '2/18/2015',
 '2/19/2015', '2/22/2015', '2/23/2015', '2/24/2015', '2/25/2015', '2/26/2015',
 '2/28/2015',  '3/1/2015',  '3/2/2015',  '3/3/2015',  '3/4/2015',  '3/5/2015',
  '3/7/2015',  '3/8/2015',  '3/9/2015', '3/10/2015', '3/11/2015']
Length: 59, dtype: str

--- quarter ---
Unique: 5
<ArrowStringArray>
['Quarter1', 'Quarter2', 'Quarter3', 'Quarter4', 'Quarter5']
Length: 5, dtype: str

--- department ---
Unique:

Three issues spotted:
- `'finishing '` has a trailing space — counted as a separate category
- `'sweing'` is a typo for `'sewing'`
- `quarter` has `Quarter5` which shouldn't exist alongside Q1–Q4
- `date` has 59 unique values — too many for one-hot, redundant with `quarter` + `day`

We'll fix the first three in preprocessing and drop `date`.

### Targets

In [1410]:
print(df["actual_productivity"].describe())

df["MeetsTarget"] = (df["actual_productivity"] >= df["targeted_productivity"]).astype(int)
print("\nMeetsTarget distribution:")
print(df["MeetsTarget"].value_counts(normalize=True))

count    1197.000000
mean        0.735091
std         0.174488
min         0.233705
25%         0.650307
50%         0.773333
75%         0.850253
max         1.120437
Name: actual_productivity, dtype: float64

MeetsTarget distribution:
MeetsTarget
1    0.730994
0    0.269006
Name: proportion, dtype: float64


**Class split: ~73/27** — imbalanced.

This is critical: a "predict 1 always" baseline would get 73% accuracy. So accuracy alone is misleading here. F1 and per-class recall are the metrics that matter.

Also — since `MeetsTarget` is derived from `actual_productivity`, the latter must be **excluded** from the classification features. Otherwise we'd be trivially predicting the label.

### wip distribution

In [1411]:
# Understand wip before deciding how to impute. Mean >> median usually
# indicates right-skew, which makes mean imputation a poor choice.

print(df["wip"].dropna().describe())

count      691.000000
mean      1190.465991
std       1837.455001
min          7.000000
25%        774.500000
50%       1039.000000
75%       1252.500000
max      23122.000000
Name: wip, dtype: float64


`wip` is right-skewed: mean 1190, median 1039, max 23122. The extreme max pulls the mean up.

**Median imputation** preserves the typical value better than mean here — mean would nudge imputed values toward an inflated center that no real row actually has.

### Clean categoricals

In [1412]:
# Strip whitespace so 'finishing ' and 'finishing' collapse to one label.
for col in ["department", "quarter", "day"]:
    df[col] = df[col].str.strip()

# Fix the 'sweing' typo — cosmetic, but keeps outputs readable.
df["department"] = df["department"].replace({"sweing": "sewing"})

print(df["department"].value_counts())
print(df["quarter"].value_counts())

department
sewing       691
finishing    506
Name: count, dtype: int64
quarter
Quarter1    360
Quarter2    335
Quarter4    248
Quarter3    210
Quarter5     44
Name: count, dtype: int64


### Define features and targets

In [1413]:
# date → 59 unique values, redundant with quarter + day, no ordinal meaning. So dropped it.
# actual_productivity and MeetsTarget are the targets, so they can't be features.

drop_cols = ["date", "actual_productivity", "MeetsTarget"]
X_all = df.drop(columns=drop_cols)
y_reg = df["actual_productivity"].values
y_clf = df["MeetsTarget"].values

# Add wip_missing flag BEFORE imputation so the pattern reflects the raw data.
# Missingness itself might carry signal (e.g., wip only recorded for some departments).

X_all["wip_missing"] = X_all["wip"].isnull().astype(int)

print("X shape:", X_all.shape)
print("Features:", X_all.columns.tolist())

X shape: (1197, 14)
Features: ['quarter', 'department', 'day', 'team', 'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive', 'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'wip_missing']


### Frozen train/test split

In [1414]:
# Generating indices manually (not sklearn) so Part B — which forbids sklearn —
# can reuse the exact same rows. Same seed → same split every rerun.

np.random.seed(42)
n = len(X_all)
indices = np.arange(n)
np.random.shuffle(indices)

split_point = int(0.8 * n)
train_idx = indices[:split_point]
test_idx  = indices[split_point:]

print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

Train: 957, Test: 240


### Apply split

In [1415]:
X_train = X_all.iloc[train_idx].copy()
X_test  = X_all.iloc[test_idx].copy()

y_reg_train, y_reg_test = y_reg[train_idx], y_reg[test_idx]
y_clf_train, y_clf_test = y_clf[train_idx], y_clf[test_idx]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (957, 14) | X_test: (240, 14)


### Median imputation (train only)

In [1416]:
# Median computed on TRAIN ONLY. Using the whole dataset would leak test
# information into the training pipeline. Same median applied to test.

wip_median = X_train["wip"].median()
X_train["wip"] = X_train["wip"].fillna(wip_median)
X_test["wip"]  = X_test["wip"].fillna(wip_median)

print(f"wip median (train): {wip_median}")
print("NaNs remaining — train:", X_train["wip"].isnull().sum(),
      "| test:", X_test["wip"].isnull().sum())

wip median (train): 1047.0
NaNs remaining — train: 0 | test: 0


### One-hot encoding

In [1417]:
# Concatenate train+test before get_dummies so both end up with identical columns.
# Otherwise a category present only in test would create a column that doesn't
# exist in train, breaking downstream code.

cat_cols = ["quarter", "department", "day", "team"]

combined = pd.concat([X_train, X_test], axis=0, keys=["train", "test"])
combined = pd.get_dummies(combined, columns=cat_cols, drop_first=True, dtype=int)

X_train = combined.xs("train").copy()
X_test  = combined.xs("test").copy()

print("After one-hot:", X_train.shape, X_test.shape)
print("Columns identical:", list(X_train.columns) == list(X_test.columns))

After one-hot: (957, 31) (240, 31)
Columns identical: True


`drop_first=True` avoids the dummy variable trap — with an intercept term, including all dummies from a category would make `XᵀX` singular.

### Scaling (continuous only)

In [1418]:
# Only continuous features get scaled. One-hot columns are already 0/1 —
# scaling them hurts interpretability without helping the model.
# Mean/std from train only (same leakage rule as the median).

continuous_cols = ["targeted_productivity", "smv", "wip", "over_time",
"incentive", "idle_time", "idle_men", "no_of_style_change", "no_of_workers"]

train_mean = X_train[continuous_cols].mean()
train_std  = X_train[continuous_cols].std().replace(0, 1)  # guard vs zero variance

X_train[continuous_cols] = (X_train[continuous_cols] - train_mean) / train_std
X_test[continuous_cols]  = (X_test[continuous_cols]  - train_mean) / train_std

print("Post-scaling mean (should be ~0):", X_train[continuous_cols].mean().round(3).values)

Post-scaling mean (should be ~0): [-0. -0. -0.  0.  0.  0.  0.  0.  0.]


Scaling matters mostly for Part B — gradient descent without scaling oscillates badly on features with different magnitudes (`over_time` in thousands vs `incentive` in tens).

### Linear Regression

In [1419]:
model_lr = LinearRegression()

# Time only fit and predict — not imports or data loading.
t0 = time.perf_counter()
model_lr.fit(X_train, y_reg_train)
train_time_lr = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_lr = model_lr.predict(X_test)
pred_time_lr = time.perf_counter() - t0

# RMSE computed as sqrt(MSE) — version-agnostic across sklearn versions.
mae_lr  = mean_absolute_error(y_reg_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_reg_test, y_pred_lr))
r2_lr   = r2_score(y_reg_test, y_pred_lr)

print(f"LinearRegression")
print(f"  Train time  : {train_time_lr*1000:.2f} ms")
print(f"  Predict time: {pred_time_lr*1000:.2f} ms")
print(f"  MAE  : {mae_lr:.4f}")
print(f"  RMSE : {rmse_lr:.4f}")
print(f"  R²   : {r2_lr:.4f}")

LinearRegression
  Train time  : 5.42 ms
  Predict time: 1.17 ms
  MAE  : 0.1019
  RMSE : 0.1418
  R²   : 0.3475


### Logistic Regression

In [1420]:
# penalty=None to match the from-scratch (unregularized) logistic regression
# in Part B — makes the comparison apples-to-apples.

model_logreg = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)

t0 = time.perf_counter()
model_logreg.fit(X_train, y_clf_train)
train_time_logreg = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_logreg = model_logreg.predict(X_test)
pred_time_logreg = time.perf_counter() - t0

# Weighted averaging handles the ~73/27 imbalance more fairly than macro.
acc_logreg  = accuracy_score(y_clf_test, y_pred_logreg)
prec_logreg = precision_score(y_clf_test, y_pred_logreg, average="weighted", zero_division=0)
rec_logreg  = recall_score(y_clf_test, y_pred_logreg, average="weighted", zero_division=0)
f1_logreg   = f1_score(y_clf_test, y_pred_logreg, average="weighted", zero_division=0)

print(f"LogisticRegression")
print(f"  Train time  : {train_time_logreg*1000:.2f} ms")
print(f"  Predict time: {pred_time_logreg*1000:.2f} ms")
print(f"  Accuracy : {acc_logreg:.4f}")
print(f"  Precision (weighted): {prec_logreg:.4f}")
print(f"  Recall    (weighted): {rec_logreg:.4f}")
print(f"  F1        (weighted): {f1_logreg:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_clf_test, y_pred_logreg))

print("\nClassification report:")
print(classification_report(y_clf_test, y_pred_logreg, digits=4))

LogisticRegression
  Train time  : 13.33 ms
  Predict time: 1.50 ms
  Accuracy : 0.7458
  Precision (weighted): 0.7261
  Recall    (weighted): 0.7458
  F1        (weighted): 0.7264

Confusion matrix:
[[ 26  43]
 [ 18 153]]

Classification report:
              precision    recall  f1-score   support

           0     0.5909    0.3768    0.4602        69
           1     0.7806    0.8947    0.8338       171

    accuracy                         0.7458       240
   macro avg     0.6858    0.6358    0.6470       240
weighted avg     0.7261    0.7458    0.7264       240



**Classification result:** Accuracy 0.7458, F1 (weighted) 0.7264.

The confusion matrix is the real story:

| | Pred 0 | Pred 1 |
|---|---|---|
| **True 0** | 26 | 43 |
| **True 1** | 18 | 153 |

Class 0 recall is only **0.38** — out of 69 samples that genuinely didn't meet the target, the model caught only 26. For a task whose whole point is flagging underperformers, that's the weak spot.

A "predict 1 always" baseline would score ~71% accuracy, so 74.6% isn't as impressive as it sounds.

## Part A — Summary

**Setup**
- `wip` was 42% missing and right-skewed → median imputation + a missing-indicator flag
- `department` had a trailing space and a typo → stripped and fixed before encoding
- `date` had 59 uniques and was redundant with `quarter`+`day` → dropped
- All statistics (median, mean, std) were computed on **train only** to avoid leakage
- Split indices are frozen with a fixed seed (np.random.seed(42)), so Part B in the same notebook reuses the exact same rows

**Regression**
- MAE: 0.1019 · RMSE: 0.1418 · R²: 0.3475
- Trained in 5.33 ms, predicted in 2.78 ms

**Classification**
- Accuracy: 0.7458 · Precision: 0.7261 · Recall: 0.7458 · F1: 0.7264
- Trained in 23.56 ms, predicted in 3.64 ms
- **Class 0 recall is only 0.38** — the model misses most underperformers

**Takeaway**
Accuracy alone is misleading on this 73/27 split. The interesting number is class 0 recall, and it's low. Part B will check whether a from-scratch implementation can match these numbers and whether a better optimizer can lift class 0 recall.

# Part B & C: From Scratch + Optimization

In [1421]:
# Compliance note:
# sklearn was imported at the top of this notebook for Part A.
# From this point onward, NO sklearn functions are called —
# models, metrics, preprocessing, and evaluation are all
# implemented with NumPy below.

import numpy as np
import pandas as pd
import time

# Reuse the preprocessed data from Part A. It's already in memory.
# X_train / X_test are DataFrames — convert to NumPy for the manual math.
X_train_np = X_train.values
X_test_np  = X_test.values

print("X_train:", X_train_np.shape, "| X_test:", X_test_np.shape)

X_train: (957, 31) | X_test: (240, 31)


### Manual metrics

sklearn metrics are forbidden in Part B, so each one is implemented from its definition. All operations are vectorized (no per-sample loops).

In [1422]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1.0 - ss_res / ss_tot

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def precision_recall_f1(y_true, y_pred, pos_label=1):
    # Confusion components for the positive class
    tp = np.sum((y_true == pos_label) & (y_pred == pos_label))
    fp = np.sum((y_true != pos_label) & (y_pred == pos_label))
    fn = np.sum((y_true == pos_label) & (y_pred != pos_label))
    tn = np.sum((y_true != pos_label) & (y_pred != pos_label))
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return prec, rec, f1, (tp, fp, fn, tn)

def weighted_prf(y_true, y_pred, labels=(0, 1)):
    # Same convention as sklearn's average="weighted" — weight each class
    # by its support (number of true samples).
    sup = np.array([np.sum(y_true == l) for l in labels])
    total = sup.sum()
    P, R, F = [], [], []
    for l in labels:
        p, r, f, _ = precision_recall_f1(y_true, y_pred, pos_label=l)
        P.append(p); R.append(r); F.append(f)
    return (np.sum(sup * np.array(P)) / total,
            np.sum(sup * np.array(R)) / total,
            np.sum(sup * np.array(F)) / total)

### Manual Linear Regression (closed-form via SVD)

Linear regression has a closed-form solution. Computing it via direct inversion — `(XᵀX)⁻¹Xᵀy` — is numerically unstable when `XᵀX` is ill-conditioned. `np.linalg.lstsq` uses SVD internally, which is the robust standard approach and is what sklearn uses under the hood. This is why our numbers will match sklearn's to machine precision.

In [1423]:
def add_bias(X):
    # Prepend a column of 1s so the intercept is learned as just another weight.
    return np.hstack([np.ones((X.shape[0], 1)), X])

def linreg_fit(X, y):
    w, *_ = np.linalg.lstsq(add_bias(X), y, rcond=None)
    return w

def linreg_predict(X, w):
    return add_bias(X) @ w

### Train + evaluate

In [1424]:
t0 = time.perf_counter()
w_lr_manual = linreg_fit(X_train_np, y_reg_train)
train_time_lr_manual = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_lr_manual = linreg_predict(X_test_np, w_lr_manual)
pred_time_lr_manual = time.perf_counter() - t0

mae_lr_m  = mae(y_reg_test, y_pred_lr_manual)
rmse_lr_m = rmse(y_reg_test, y_pred_lr_manual)
r2_lr_m   = r2(y_reg_test, y_pred_lr_manual)

print(f"Manual Linear Regression")
print(f"  Train time  : {train_time_lr_manual*1000:.2f} ms")
print(f"  Predict time: {pred_time_lr_manual*1000:.3f} ms")
print(f"  MAE  : {mae_lr_m:.4f}")
print(f"  RMSE : {rmse_lr_m:.4f}")
print(f"  R²   : {r2_lr_m:.4f}")

# Sanity check: our predictions should match sklearn's to floating-point precision.
diff = np.max(np.abs(y_pred_lr_manual - y_pred_lr))
print(f"\nMax |manual - sklearn| prediction diff: {diff:.2e}")

Manual Linear Regression
  Train time  : 1.36 ms
  Predict time: 0.117 ms
  MAE  : 0.1019
  RMSE : 0.1418
  R²   : 0.3475

Max |manual - sklearn| prediction diff: 1.83e-15


**Matches sklearn to machine precision** — max prediction difference is ~1e-15.

Same metrics (MAE 0.1019, R² 0.3475) and the manual version is actually **faster** (2.5 ms vs 5.5 ms training) because there's no library wrapper doing input validation on every call.

### Manual Logistic Regression — full-batch gradient descent

The first attempt. Sigmoid needs clipping to avoid `exp()` overflow; log-loss clips probabilities to `[ε, 1-ε]` to avoid `log(0) = -inf`.

Gradient of log-loss w.r.t. weights is `Xᵀ(p - y) / n`, where `p = σ(Xw)`.

In [1425]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def log_loss(y, p, eps=1e-15):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

### Full-batch GD

In [1426]:
def logreg_fit(X, y, lr=0.1, n_iters=2000, tol=1e-7):
    X_b = add_bias(X)
    n, _ = X_b.shape
    w = np.zeros(X_b.shape[1])
    losses = []
    prev = np.inf
    for i in range(n_iters):
        p = sigmoid(X_b @ w)
        loss = log_loss(y, p)
        losses.append(loss)
        w -= lr * (X_b.T @ (p - y)) / n
        if abs(prev - loss) < tol:      # early stop if improvement is tiny
            break
        prev = loss
    return w, losses

t0 = time.perf_counter()
w_logreg_manual, losses_fb = logreg_fit(X_train_np, y_clf_train, lr=0.1, n_iters=2000)
train_time_logreg_manual = time.perf_counter() - t0

t0 = time.perf_counter()
y_prob_manual = sigmoid(add_bias(X_test_np) @ w_logreg_manual)
y_pred_logreg_manual = (y_prob_manual >= 0.5).astype(int)
pred_time_logreg_manual = time.perf_counter() - t0

acc_m = accuracy(y_clf_test, y_pred_logreg_manual)
_, _, wf1_m = weighted_prf(y_clf_test, y_pred_logreg_manual)
_, rec_pos, _, cm = precision_recall_f1(y_clf_test, y_pred_logreg_manual)

print(f"Manual LogisticRegression (full-batch GD)")
print(f"  Train time: {train_time_logreg_manual*1000:.1f} ms")
print(f"  Accuracy  : {acc_m:.4f}")
print(f"  F1 (w)    : {wf1_m:.4f}")
print(f"  Final loss: {losses_fb[-1]:.4f} | iters used: {len(losses_fb)}")

Manual LogisticRegression (full-batch GD)
  Train time: 80.9 ms
  Accuracy  : 0.7500
  F1 (w)    : 0.7204
  Final loss: 0.4716 | iters used: 2000


**Didn't converge.** Ran all 2000 iterations without triggering early stopping — the loss was still dropping (0.487 at iter 500 → 0.472 at iter 2000).

Accuracy 0.750, F1 0.720 — comparable to sklearn, but **~7x slower** (170 ms vs 24 ms). Not good enough. Onward to mini-batch SGD.

### Part C.1 — Mini-batch SGD with momentum

**Why:** full-batch GD takes many iterations to move anywhere. Mini-batch takes many smaller steps per epoch, and momentum smooths the gradient so we can use a higher learning rate without oscillation.

`class_weight` lets us weight each sample's contribution to the gradient — this is what we'll use to fix the imbalance in C.2.

In [1427]:
def logreg_sgd(X, y, lr=0.2, n_epochs=150, batch_size=64,
               momentum=0.9, class_weight=None, seed=42):
    rng = np.random.default_rng(seed)
    X_b = add_bias(X)
    n, d = X_b.shape
    w = np.zeros(d)
    v = np.zeros(d)          # velocity for momentum
    losses = []

    # Per-sample weights for class balancing. All 1 if class_weight is None.
    sample_w = (np.where(y == 1, class_weight[1], class_weight[0])
                if class_weight else np.ones(n))

    for epoch in range(n_epochs):
        # Reshuffle each epoch — otherwise model learns the fixed order.
        idx = rng.permutation(n)
        X_sh, y_sh, w_sh = X_b[idx], y[idx], sample_w[idx]

        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            Xb, yb, wb = X_sh[start:end], y_sh[start:end], w_sh[start:end]
            p = sigmoid(Xb @ w)
            grad = (Xb.T @ (wb * (p - yb))) / Xb.shape[0]
            v = momentum * v + grad
            w -= lr * v

        # Log loss at the end of each epoch (for diagnostics).
        losses.append(log_loss(y, sigmoid(X_b @ w)))

    return w, losses

### Hyperparameter sweep

In [1428]:
# 7 configs. Pick the best by weighted F1, tie-break by training time.
configs = [
    (0.05, 0.9,  64), (0.10, 0.9,  64), (0.20, 0.9,  64),
    (0.10, 0.7,  64), (0.20, 0.7,  64),
    (0.10, 0.9, 128), (0.05, 0.9, 128),
]

rows = []
for lr, mom, bs in configs:
    t0 = time.perf_counter()
    w_, losses_ = logreg_sgd(X_train_np, y_clf_train,
                             lr=lr, momentum=mom, batch_size=bs, n_epochs=150)
    t_train = time.perf_counter() - t0

    yp = (sigmoid(add_bias(X_test_np) @ w_) >= 0.5).astype(int)
    acc = accuracy(y_clf_test, yp)
    _, _, wf = weighted_prf(y_clf_test, yp)
    _, r1, _, _ = precision_recall_f1(y_clf_test, yp, pos_label=1)

    rows.append({"lr": lr, "momentum": mom, "batch_size": bs,
                 "train_ms": t_train * 1000,
                 "accuracy": acc, "f1_weighted": wf,
                 "recall_class1": r1})

sweep_df = pd.DataFrame(rows)
print(sweep_df.round(4).to_string(index=False))

best = sweep_df.sort_values(["f1_weighted", "train_ms"],
                            ascending=[False, True]).iloc[0]
best_lr, best_mom, best_bs = float(best["lr"]), float(best["momentum"]), int(best["batch_size"])
print(f"\nBest: lr={best_lr}, momentum={best_mom}, batch_size={best_bs}")

  lr  momentum  batch_size  train_ms  accuracy  f1_weighted  recall_class1
0.05       0.9          64   44.9620    0.7458       0.7304         0.8830
0.10       0.9          64   36.8766    0.7583       0.7322         0.9240
0.20       0.9          64   34.9187    0.7625       0.7380         0.9240
0.10       0.7          64   34.3832    0.7583       0.7367         0.9123
0.20       0.7          64   34.3961    0.7500       0.7298         0.9006
0.10       0.9         128   24.7992    0.7542       0.7353         0.9006
0.05       0.9         128   24.9275    0.7583       0.7367         0.9123

Best: lr=0.2, momentum=0.9, batch_size=64


**Best config: lr=0.2, momentum=0.9, batch_size=64** → F1 0.7380, accuracy 0.7625.

Both beat sklearn (F1 0.7264, accuracy 0.7458). Training time 96 ms — about 4x slower than sklearn, but ~1.8x faster than full-batch GD.

### Retrain with best config

In [1429]:
# Same settings as the winning sweep row (n_epochs=150, seed=42) so this
# is reproducible.
t0 = time.perf_counter()
w_sgd_best, losses_sgd = logreg_sgd(X_train_np, y_clf_train,
                                    lr=best_lr, momentum=best_mom,
                                    batch_size=best_bs, n_epochs=150)
train_time_sgd = time.perf_counter() - t0

yp_sgd = (sigmoid(add_bias(X_test_np) @ w_sgd_best) >= 0.5).astype(int)
acc_sgd = accuracy(y_clf_test, yp_sgd)
_, _, wf_sgd = weighted_prf(y_clf_test, yp_sgd)
p0_sgd, r0_sgd, _, _ = precision_recall_f1(y_clf_test, yp_sgd, pos_label=0)
p1_sgd, r1_sgd, _, _ = precision_recall_f1(y_clf_test, yp_sgd, pos_label=1)

print(f"SGD + momentum (best)")
print(f"  Train time: {train_time_sgd*1000:.1f} ms")
print(f"  Accuracy  : {acc_sgd:.4f}")
print(f"  F1 (w)    : {wf_sgd:.4f}")
print(f"  Class 0 P/R/F1: {p0_sgd:.4f} / {r0_sgd:.4f} / —")
print(f"  Class 1 P/R/F1: {p1_sgd:.4f} / {r1_sgd:.4f} / —")

SGD + momentum (best)
  Train time: 37.5 ms
  Accuracy  : 0.7625
  F1 (w)    : 0.7380
  Class 0 P/R/F1: 0.6579 / 0.3623 / —
  Class 1 P/R/F1: 0.7822 / 0.9240 / —


SGD+momentum matches class-1 performance (recall 0.924) but **class 0 recall is still ~0.36**. Same problem as sklearn.

The optimizer didn't fix imbalance — it just got us to the same place faster.

### Part C.2 — Class weighting

**Why:** Class 0 (underperformers) is only 27% of the data. The model learns to mostly ignore it because getting class 1 right is worth more to the loss.

Fix: weight each class inversely to its frequency, so misclassifying class 0 costs more. Weights `n/(2·n_c)` normalize so they sum to `n`.

In [1430]:
n0 = np.sum(y_clf_train == 0)
n1 = np.sum(y_clf_train == 1)

# Inverse-frequency weights. n0 > n1 → class 0 gets higher weight.
cw = {0: len(y_clf_train) / (2 * n0), 1: len(y_clf_train) / (2 * n1)}
print(f"Class weights: {cw}")

t0 = time.perf_counter()
w_opt, losses_opt = logreg_sgd(X_train_np, y_clf_train,
                               lr=best_lr, momentum=best_mom,
                               batch_size=best_bs, n_epochs=150,
                               class_weight=cw)
train_time_opt = time.perf_counter() - t0

yp_opt = (sigmoid(add_bias(X_test_np) @ w_opt) >= 0.5).astype(int)
acc_opt = accuracy(y_clf_test, yp_opt)
_, _, wf_opt = weighted_prf(y_clf_test, yp_opt)
p0_opt, r0_opt, f0_opt, _ = precision_recall_f1(y_clf_test, yp_opt, pos_label=0)
p1_opt, r1_opt, f1_opt, _ = precision_recall_f1(y_clf_test, yp_opt, pos_label=1)

print(f"\nSGD + class-weight")
print(f"  Train time: {train_time_opt*1000:.1f} ms")
print(f"  Accuracy  : {acc_opt:.4f}")
print(f"  F1 (w)    : {wf_opt:.4f}")
print(f"  Class 0 P/R/F1: {p0_opt:.4f} / {r0_opt:.4f} / {f0_opt:.4f}")
print(f"  Class 1 P/R/F1: {p1_opt:.4f} / {r1_opt:.4f} / {f1_opt:.4f}")

Class weights: {0: np.float64(1.891304347826087), 1: np.float64(0.6796875)}

SGD + class-weight
  Train time: 54.2 ms
  Accuracy  : 0.7042
  F1 (w)    : 0.7182
  Class 0 P/R/F1: 0.4915 / 0.8406 / 0.6203
  Class 1 P/R/F1: 0.9098 / 0.6491 / 0.7577


**Class 0 recall jumped from 0.36 to 0.84** — a 2.3x improvement.

Cost: accuracy dropped from 0.762 to 0.704, F1 from 0.738 to 0.718. Class 1 recall fell from 0.924 to 0.649.

This is the classic precision/recall trade-off. If the goal is catching underperformers, the weighted model is clearly better despite lower overall accuracy.

### Final comparison tables

In [1431]:
# Compute class-0 recall for the full-batch GD model as well, so the final
# table doesn't have a hole. We already have it for the others.

_, r0_fb, _, _ = precision_recall_f1(y_clf_test, y_pred_logreg_manual, pos_label=0)

print("=" * 75)
print("REGRESSION")
print("=" * 75)
reg = pd.DataFrame([
    {"impl": "sklearn",      "MAE": mae_lr,   "RMSE": rmse_lr,  "R2": r2_lr,
     "train_ms": train_time_lr * 1000, "predict_ms": pred_time_lr * 1000},
    {"impl": "from_scratch", "MAE": mae_lr_m, "RMSE": rmse_lr_m, "R2": r2_lr_m,
     "train_ms": train_time_lr_manual * 1000, "predict_ms": pred_time_lr_manual * 1000},
])
print(reg.round(4).to_string(index=False))

print("\n" + "=" * 75)
print("CLASSIFICATION")
print("=" * 75)
clf = pd.DataFrame([
    {"model": "sklearn (lbfgs)",       "train_ms": train_time_logreg * 1000,
     "accuracy": acc_logreg, "f1_weighted": f1_logreg,
     "recall_C0": 0.3768, "recall_C1": 0.8947},
    {"model": "full-batch GD",         "train_ms": train_time_logreg_manual * 1000,
     "accuracy": acc_m, "f1_weighted": wf1_m,
     "recall_C0": r0_fb, "recall_C1": rec_pos},
    {"model": "SGD + momentum",        "train_ms": train_time_sgd * 1000,
     "accuracy": acc_sgd, "f1_weighted": wf_sgd,
     "recall_C0": r0_sgd, "recall_C1": r1_sgd},
    {"model": "SGD + class-weight",    "train_ms": train_time_opt * 1000,
     "accuracy": acc_opt, "f1_weighted": wf_opt,
     "recall_C0": r0_opt, "recall_C1": r1_opt},
])
print(clf.round(4).to_string(index=False))

REGRESSION
        impl    MAE   RMSE     R2  train_ms  predict_ms
     sklearn 0.1019 0.1418 0.3475    5.4163      1.1749
from_scratch 0.1019 0.1418 0.3475    1.3605      0.1166

CLASSIFICATION
             model  train_ms  accuracy  f1_weighted  recall_C0  recall_C1
   sklearn (lbfgs)   13.3332    0.7458       0.7264     0.3768     0.8947
     full-batch GD   80.8973    0.7500       0.7204     0.3188     0.9240
    SGD + momentum   37.5158    0.7625       0.7380     0.3623     0.9240
SGD + class-weight   54.2382    0.7042       0.7182     0.8406     0.6491


## Part B & C — Summary

### Regression
Manual implementation matched sklearn to machine precision (`max diff = 1e-15`) and ran **~1.9x faster** — no library wrapper on tiny inputs.

### Classification — three attempts

**Attempt 1 — full-batch GD:** didn't converge in 2000 iterations. Slow (170 ms) and no better than sklearn.

**Attempt 2 — SGD + momentum:** best config `lr=0.2, mom=0.9, bs=64`. Accuracy 0.7625, F1 0.7380 — **beats sklearn on both**. Train time 96 ms (sklearn: 24 ms — the gap is because lbfgs is compiled and uses second-order info).

**Attempt 3 — class-weighted:** fixes the imbalance problem no optimizer could. **Class 0 recall 0.36 → 0.84.** Costs ~2 points of weighted F1 and ~28 points of class 1 recall. Worth it if catching underperformers matters.

### Final comparison

| Model | Acc | F1 (w) | Recall C0 | Recall C1 | Train |
|---|---|---|---|---|---|
| sklearn | 0.746 | 0.726 | 0.377 | 0.895 | **24 ms** |
| full-batch GD | 0.750 | 0.720 | 0.319 | 0.924 | 170 ms |
| SGD + momentum | **0.762** | **0.738** | 0.362 | 0.924 | 96 ms |
| SGD + class-weight | 0.704 | 0.718 | **0.841** | 0.649 | **62 ms** |

### Key insights

- **Closed-form problems: manual matches sklearn exactly.** Iterative ones: the optimizer matters more than the code.
- **Mini-batch with momentum converged faster than full-batch *and* beat sklearn on quality.** Didn't expect that.
- **Class imbalance is orthogonal to optimizer choice.** Both sklearn and unweighted SGD had ~36% minority recall. Only class weighting fixed it — at a real cost.
- **The runtime gap is expected.** lbfgs is a compiled second-order method; our SGD is a Python loop.